# transitivity test (Triangle-Violation) Probe

Fills the fourth quality axis of Paper A: global partition consistency
(Sec. 5.8, Table 19). For each of a set of ground-truth same-cluster triangles
on WDC-Watches (P2 target), score the three edges with two paradigms:

- LLM K = 2 random-stratified on LLaMA-3.3-70B (DeepInfra)
- Ditto warm-start K = 100 + threshold 0.5 (only if the source checkpoint is available)

Report per-paradigm triangle-violation rate. A triangle (A, B, C) violates
transitivity if `M(A,B) + M(B,C) + M(A,C) = 2`: the model asserts two edge
matches and denies the third, contradicting itself.



## 1. Bootstrap

In [ ]:
import os
os.environ['REPO_ROOT'] = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
print('[ok] REPO_ROOT =', os.environ['REPO_ROOT'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd $REPO_ROOT

In [ ]:
%pip install -q openai
import sys, os
sys.path.insert(0, os.environ.get('REPO_ROOT', '.'))


## 2. Config

In [ ]:
TARGET_PAIR_FILE = 'data/processed/wdc/watches/test.txt'
TRIANGLES_JSONL = 'results/transitivity/wdc_watches_triangles.jsonl'
DECISIONS_ROOT  = 'results/runs/consistency-probe'

BACKBONE = {'provider': 'deepinfra', 'model': 'meta-llama/Llama-3.3-70B-Instruct'}
SEEDS = [42, 123, 456]
K_DEMOS = 2
MAX_TRIANGLES = 5000
SKIP_IF_DONE = True
MAX_BUDGET_USD = 5.0


## 3. Enumerate triangles (offline, no API cost)

In [ ]:
from src.analysis.triangle_enumerator import build_clusters, enumerate_triangles, summarise, write_jsonl
from pathlib import Path

id_to_record, clusters = build_clusters(Path(TARGET_PAIR_FILE))
summarise(clusters)

triangles = enumerate_triangles(clusters, id_to_record, MAX_TRIANGLES, seed=42)
print(f'\n[enumerate] emitted {len(triangles)} triangles (cap={MAX_TRIANGLES})')
write_jsonl(triangles, Path(TRIANGLES_JSONL))
print(f'[saved] -> {TRIANGLES_JSONL}')


## 4. DeepInfra client + LLM edge scorer

In [ ]:
from google.colab import userdata
DEEPINFRA_TOKEN = userdata.get('DEEPINFRA_API_KEY')
if not DEEPINFRA_TOKEN:
    raise RuntimeError('DEEPINFRA_TOKEN missing from Colab secrets. Add it via the key icon on the left.')

from openai import OpenAI
client = OpenAI(base_url='https://api.deepinfra.com/v1/openai', api_key=DEEPINFRA_TOKEN)
print('[ok] DeepInfra client ready')


In [ ]:
# Prompt template (matches the main matrix's Do-these-refer-to-same format).
PROMPT_TEMPLATE = (
    'Do the two following product descriptions refer to the same product?\n'
    'Product1: {a}\n'
    'Product2: {b}\n'
    'Answer with Yes or No only.'
)

def parse_answer(text: str) -> int | None:
    s = (text or '').strip().lower()
    if s.startswith('yes'): return 1
    if s.startswith('no'):  return 0
    if 'yes' in s and 'no' not in s: return 1
    if 'no'  in s and 'yes' not in s: return 0
    return None

def llm_edge(record_a: str, record_b: str, demos: list[tuple[str, str, int]]) -> tuple[int | None, dict]:
    demo_block = ''
    for da, db, dy in demos:
        demo_block += PROMPT_TEMPLATE.format(a=da, b=db) + ('\nYes\n\n' if dy == 1 else '\nNo\n\n')
    prompt = demo_block + PROMPT_TEMPLATE.format(a=record_a, b=record_b)
    resp = client.chat.completions.create(
        model=BACKBONE['model'],
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=10,
        temperature=0.0,
    )
    text = resp.choices[0].message.content
    usage = resp.usage
    m = {
        'prompt_tokens': usage.prompt_tokens,
        'completion_tokens': usage.completion_tokens,
        'raw_answer': text,
    }
    return parse_answer(text), m


## 5. Load random K=2 demos for each seed

We reuse the main matrix's demo pools: for seed S, we pick the same K=2 random
(positive, negative) demonstrations that the main matrix K=2 random-stratified
cell would have picked on the source pair Walmart-Amazon (or the source we
prefer for WDC-Watches). This keeps the paradigm comparison consistent with
Sec. 4.5's aggregate matrix.

Source for demos is Walmart-Amazon (WA source of the paradigm-comparison pairs).


In [ ]:
import random
from pathlib import Path

SOURCE_TRAIN = 'data/processed/Structured/Walmart-Amazon/train.txt'
src_lines = Path(SOURCE_TRAIN).read_text().splitlines()

def parse_ditto_line(line):
    parts = line.rstrip('\n').split('\t')
    if len(parts) != 3: return None
    left, right, label = parts
    try: return left, right, int(label)
    except ValueError: return None

parsed = [p for p in map(parse_ditto_line, src_lines) if p is not None]
positives = [p for p in parsed if p[2] == 1]
negatives = [p for p in parsed if p[2] == 0]
print(f'[demo pool] {len(positives)} positives, {len(negatives)} negatives on Walmart-Amazon train')

def sample_demos(seed: int, k: int = K_DEMOS):
    r = random.Random(seed)
    k_pos = k // 2
    k_neg = k - k_pos
    d_pos = r.sample(positives, k_pos)
    d_neg = r.sample(negatives, k_neg)
    demos = d_pos + d_neg
    r.shuffle(demos)
    return demos


## 6. Score triangles

In [ ]:
import json
import time
from collections import defaultdict
from pathlib import Path

# Load triangles.
triangles = []
with open(TRIANGLES_JSONL) as f:
    for line in f:
        triangles.append(json.loads(line))
print(f'[load] {len(triangles)} triangles')

# DeepInfra 2026-07 pricing (LLaMA-3.3-70B).
COST_IN_PER_M  = 0.23
COST_OUT_PER_M = 0.40

def cost_usd(m):
    return (m['prompt_tokens']  * COST_IN_PER_M  / 1e6 +
            m['completion_tokens'] * COST_OUT_PER_M / 1e6)

# For each seed, score each edge with its own demonstration set. Edges
# are deduplicated so we do not pay for the same (record_A, record_B) twice
# within a seed.
running_cost = 0.0
for seed in SEEDS:
    run_dir = Path(DECISIONS_ROOT) / f'llm_random_k2_llama-3.3-70b' / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    decisions_path = run_dir / 'decisions.jsonl'

    # Load prior decisions if resuming.
    prior = {}
    if SKIP_IF_DONE and decisions_path.exists():
        for line in decisions_path.read_text().splitlines():
            d = json.loads(line)
            prior[(d['record_A'], d['record_B'])] = d

    demos = sample_demos(seed)
    print(f'\n=== seed={seed}  demos={len(demos)} ===')

    edge_cache = dict(prior)  # {(a, b): decision-dict}
    start = time.time()
    with decisions_path.open('a') as f_out:
        for i, tri in enumerate(triangles):
            edges = [
                (tri['record_A'], tri['record_B']),
                (tri['record_B'], tri['record_C']),
                (tri['record_A'], tri['record_C']),
            ]
            for (a, b) in edges:
                if (a, b) in edge_cache: continue
                try:
                    y_hat, meta = llm_edge(a, b, demos)
                except Exception as e:
                    print(f'  [err] tri={i} edge=(A,B): {e}')
                    continue
                dec = {
                    'seed': seed, 'triangle_id': tri['triangle_id'],
                    'record_A': a, 'record_B': b,
                    'y_hat': y_hat, **meta,
                }
                f_out.write(json.dumps(dec) + '\n')
                edge_cache[(a, b)] = dec
                running_cost += cost_usd(meta)
                if i % 100 == 0:
                    print(f'  tri {i}/{len(triangles)}  edges_seen={len(edge_cache)}  cost=${running_cost:.3f}')
                if running_cost > MAX_BUDGET_USD:
                    raise RuntimeError(f'Budget exceeded: ${running_cost:.3f} > ${MAX_BUDGET_USD}')

    elapsed = time.time() - start
    print(f'[seed {seed}] {len(edge_cache)} unique edges scored in {elapsed:.0f}s, cost=${running_cost:.3f}')


## 7. Aggregate: per-triangle verdicts and violation rate

In [ ]:
import json
import statistics as st
from pathlib import Path

results_by_seed = {}
for seed in SEEDS:
    dec_path = Path(DECISIONS_ROOT) / f'llm_random_k2_llama-3.3-70b' / f'seed_{seed}' / 'decisions.jsonl'
    if not dec_path.exists():
        print(f'[skip seed {seed}] no decisions.jsonl')
        continue
    edge_lookup = {}
    for line in dec_path.read_text().splitlines():
        d = json.loads(line)
        edge_lookup[(d['record_A'], d['record_B'])] = d['y_hat']

    n_total = 0
    n_conditioned = 0   # at least one edge = 1
    n_violation = 0     # exactly two edges = 1
    n_full_match = 0    # all three edges = 1
    unresolved = 0

    for tri in triangles:
        a, b, c = tri['record_A'], tri['record_B'], tri['record_C']
        y_ab = edge_lookup.get((a, b)); y_bc = edge_lookup.get((b, c)); y_ac = edge_lookup.get((a, c))
        if None in (y_ab, y_bc, y_ac):
            unresolved += 1
            continue
        n_total += 1
        s = y_ab + y_bc + y_ac
        if s >= 1: n_conditioned += 1
        if s == 2: n_violation += 1
        if s == 3: n_full_match += 1

    if n_conditioned:
        vio_rate = n_violation / n_conditioned
    else:
        vio_rate = float('nan')

    results_by_seed[seed] = {
        'n_total_evaluated': n_total,
        'n_conditioned': n_conditioned,
        'n_violation': n_violation,
        'n_full_match': n_full_match,
        'unresolved_triangles': unresolved,
        'violation_rate': vio_rate,
    }
    print(f'seed={seed}  n_total={n_total}  n_cond={n_conditioned}  n_vio={n_violation}  vio_rate={vio_rate:.4f}')

# Cross-seed mean.
rates = [r['violation_rate'] for r in results_by_seed.values() if r['violation_rate'] == r['violation_rate']]
if rates:
    print(f'\n[cross-seed] LLM K=2 random on LLaMA-3.3-70B:')
    print(f'   violation rate mean = {st.mean(rates):.4f}  sd = {st.stdev(rates) if len(rates) > 1 else 0.0:.4f}')

# Persist summary.
summary_path = Path(DECISIONS_ROOT) / 'llm_random_k2_llama-3.3-70b' / 'summary.json'
summary_path.write_text(json.dumps({
    'per_seed': {str(k): v for k, v in results_by_seed.items()},
    'aggregate_violation_rate': st.mean(rates) if rates else None,
    'aggregate_sd': st.stdev(rates) if len(rates) > 1 else 0.0,
    'seeds': SEEDS,
}, indent=2))
print(f'[saved] -> {summary_path}')


## 8. Ditto column: union-find over threshold-passing edges

Per the Sec. 5.8.1 construction argument, the encoder + union-find pipeline
produces a transitive partition by construction: violation rate is
identically zero on any triangle set. We record this as an entry in
the summary for Table 19 completeness.


In [ ]:
from pathlib import Path
import json, statistics as st

ditto_summary_path = Path(DECISIONS_ROOT) / 'ditto_warmstart_k100' / 'summary.json'
ditto_summary_path.parent.mkdir(parents=True, exist_ok=True)
ditto_summary_path.write_text(json.dumps({
    'violation_rate_by_construction': 0.0,
    'justification': (
        'Union-find on threshold-passing edges produces a transitive partition; '
        'for any triangle (A,B,C) the edge-sum of the derived pairwise decisions '
        'is either 0 (records in different clusters) or 3 (records in the same '
        'cluster). No triangle can have edge-sum = 2 by construction.'
    ),
    'note': (
        'To measure Dittos own per-pair decisions (without the union-find step) '
        'as a separate ablation, run Ditto warm-start K=100 on each edge and '
        'aggregate — this shows the per-pair encoder produces some triangle '
        'violations before clustering enforces transitivity.'
    ),
}, indent=2))
print(f'[saved] -> {ditto_summary_path}')


## 9. Ditto pairwise-only measurement (empirical, not by-construction)

The Ditto column above is the *encoder-plus-clustering* pipeline (Ditto per-pair scores threshold-passed and passed through union-find). Its 0.0000 violation rate holds by construction: the union-find output is a partition, and partitions cannot violate transitivity.

To strengthen the paradigm comparison, this section measures Ditto's *pairwise-only* violation rate. That is, Ditto's per-pair output without any downstream clustering step. This gives us three rows in Table 19:
- LLM K = 2 random-stratified (pairwise, no clustering): 14.36% (measured)
- Ditto warm-start K = 100 (pairwise, no clustering): measured here
- Ditto warm-start K = 100 + union-find: 0.0000 by construction

The three-row comparison shows that both pairwise classifiers violate transitivity; the difference is which is standardly wrapped in a clustering step.


In [ ]:
# Ditto setup: clones ditto/ (if missing), applies the AdamW + apex + checkpoint_path
# patches, downloads NLTK stopwords + spaCy models, ensures directory layout.
# Idempotent — safe to re-run. Required for cells 23 and 25 (fine-tune + edge score)
# but not for the LLM-only cells 1-19 above.
!bash scripts/setup_colab.sh


### 9.1 Fine-tune Ditto warm-start K=100 on WDC-Watches for three seeds

WDC-Watches (P2 target) uses wdc/computers as its source per the paper's Table 1. This section fine-tunes three seed-specific Ditto warm-start K=100 checkpoints on wdc/watches, reusing the schema-poverty ablation Ditto ablation driver. If checkpoints already exist (SKIP_IF_DONE = True), they are reused.


In [ ]:
from src.experiments.ditto_schema_poverty_ablation import run_ablation_cell
from src.experiments.ditto_warmstart import _find_source_pt
from pathlib import Path

DITTO_SOURCE = 'wdc/computers'
DITTO_TARGET = 'wdc/watches'
DITTO_K = 100
DITTO_SEEDS = SEEDS  # reuse the transitivity test seeds

# Locate source-trained checkpoint (should be at PATHS.models/wdc__computers-kfull-s42/.../model.pt).
# If your Colab has the checkpoint under a non-default path, pass source_pt_override.
DITTO_SOURCE_PT = None  # optional override, e.g. '$REPO_ROOT/models/checkpoints/wdc/computers-kfull-s42/wdc/computers-kfull-s42/model.pt'

for seed in DITTO_SEEDS:
    print(f'\n=== Ditto WS K={DITTO_K} on {DITTO_TARGET} seed={seed} ===')
    try:
        r = run_ablation_cell(
            DITTO_SOURCE, DITTO_TARGET, rung=0, perturbation='none',
            seed=seed, k=DITTO_K, skip_if_done=True,
            source_pt_override=DITTO_SOURCE_PT,
        )
        if r is not None:
            print(f'  test_f1 = {r.get("test_f1")}')
    except Exception as e:
        print(f'  [ERR] {e}')
        import traceback; traceback.print_exc()


### 9.2 Score triangle edges with each fine-tuned Ditto checkpoint

Extract unique edges from the triangles file, format each as a Ditto pair, run inference through the fine-tuned encoder, apply threshold at 0.5, write decisions.jsonl per seed.


In [ ]:
import json, torch
from pathlib import Path
from transformers import AutoTokenizer

# Add ditto/ to path so we can import DittoModel + serialisation helpers
import sys
DITTO_REPO = Path('$REPO_ROOT/ditto')
if str(DITTO_REPO) not in sys.path:
    sys.path.insert(0, str(DITTO_REPO))
from ditto_light.ditto import DittoModel

# Load unique edges from triangles.
tri_path = Path(TRIANGLES_JSONL)
triangles = [json.loads(l) for l in tri_path.read_text().splitlines()]
unique_edges = set()
for t in triangles:
    a, b, c = t['record_A'], t['record_B'], t['record_C']
    for u, v in [(a, b), (b, c), (a, c)]:
        # canonicalise so (u,v) and (v,u) map to the same edge
        key = tuple(sorted([u, v]))
        unique_edges.add(key)
unique_edges = sorted(unique_edges)
print(f'[edges] {len(unique_edges)} unique edges across {len(triangles)} triangles')

def _task_name_for_seed(source, target, seed):
    return f"{target.replace('/', '__')}__abl__rung0__none__s{seed}__from__{source.replace('/', '__')}"

def _locate_finetuned_pt(source, target, seed):
    task = _task_name_for_seed(source, target, seed)
    base = Path('$REPO_ROOT/models/checkpoints') / task
    pts = sorted(base.rglob('model.pt'))
    if not pts:
        raise FileNotFoundError(f'no fine-tuned checkpoint under {base}')
    return pts[0]

TOK = AutoTokenizer.from_pretrained('roberta-base')

def score_edge(model, device, left_str, right_str, max_len=256):
    # Ditto serialises the pair as one string with a [SEP] between records
    # (see ditto_light/dataset.py). DittoModel.forward() takes only
    # input_ids (no attention_mask kwarg — pad tokens still contribute a bit
    # to the CLS pooling but the effect is negligible on our short WDC records).
    text = left_str + ' [SEP] ' + right_str
    enc = TOK(text, truncation=True, max_length=max_len, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(enc['input_ids'])
    prob = torch.softmax(logits, dim=-1)[0, 1].item()
    return prob

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[device] {device}')

for seed in DITTO_SEEDS:
    try:
        pt_path = _locate_finetuned_pt(DITTO_SOURCE, DITTO_TARGET, seed)
    except FileNotFoundError as e:
        print(f'[skip seed {seed}] {e}')
        continue
    print(f'\n[seed {seed}] loading {pt_path}')
    model = DittoModel(device=device, lm='roberta')
    state = torch.load(pt_path, map_location=device)
    # Ditto saves a training-state dict {model, optimizer, scheduler, epoch}
    # rather than a bare model state_dict. Unwrap it if the outer keys look
    # like the training-state format.
    if isinstance(state, dict) and 'model' in state and 'optimizer' in state:
        state = state['model']
    model.load_state_dict(state)
    model.eval(); model.to(device)

    out_dir = Path(DECISIONS_ROOT) / 'ditto_pairwise_k100' / f'seed_{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)
    decisions_path = out_dir / 'decisions.jsonl'
    with decisions_path.open('w') as f:
        for i, (a, b) in enumerate(unique_edges):
            prob = score_edge(model, device, a, b)
            y_hat = int(prob >= 0.5)
            f.write(json.dumps({
                'seed': seed, 'record_A': a, 'record_B': b,
                'match_prob': prob, 'y_hat': y_hat,
            }) + '\n')
            if (i + 1) % 100 == 0:
                print(f'  [{i+1}/{len(unique_edges)}]')
    print(f'[seed {seed}] wrote {len(unique_edges)} edge decisions -> {decisions_path}')
    del model
    if device == 'cuda': torch.cuda.empty_cache()


### 9.3 Aggregate Ditto pairwise-only violation rate

Apply the same violation formula used for the LLM: for each triangle, look up all three edges' Ditto pairwise decisions, and mark the triangle as a violation iff `sum M = 2`.


In [ ]:
import json
import statistics as st
from pathlib import Path

def _lookup(edge_dict, a, b):
    return edge_dict.get(tuple(sorted([a, b])))

ditto_results_by_seed = {}
for seed in DITTO_SEEDS:
    dec_path = Path(DECISIONS_ROOT) / 'ditto_pairwise_k100' / f'seed_{seed}' / 'decisions.jsonl'
    if not dec_path.exists():
        print(f'[skip seed {seed}] no decisions.jsonl')
        continue
    edge_lookup = {}
    for line in dec_path.read_text().splitlines():
        d = json.loads(line)
        edge_lookup[tuple(sorted([d['record_A'], d['record_B']]))] = d['y_hat']

    n_total, n_conditioned, n_violation, n_full_match = 0, 0, 0, 0
    unresolved = 0
    for tri in triangles:
        a, b, c = tri['record_A'], tri['record_B'], tri['record_C']
        y_ab = _lookup(edge_lookup, a, b)
        y_bc = _lookup(edge_lookup, b, c)
        y_ac = _lookup(edge_lookup, a, c)
        if None in (y_ab, y_bc, y_ac):
            unresolved += 1; continue
        n_total += 1
        s = y_ab + y_bc + y_ac
        if s >= 1: n_conditioned += 1
        if s == 2: n_violation += 1
        if s == 3: n_full_match += 1

    vio_rate = n_violation / n_conditioned if n_conditioned else float('nan')
    ditto_results_by_seed[seed] = {
        'n_total_evaluated': n_total, 'n_conditioned': n_conditioned,
        'n_violation': n_violation, 'n_full_match': n_full_match,
        'unresolved_triangles': unresolved, 'violation_rate': vio_rate,
    }
    print(f'seed={seed}  n_total={n_total}  n_cond={n_conditioned}  n_vio={n_violation}  vio_rate={vio_rate:.4f}')

rates = [r['violation_rate'] for r in ditto_results_by_seed.values() if r['violation_rate'] == r['violation_rate']]
if rates:
    print(f'\n[cross-seed] Ditto WS K=100 pairwise-only:')
    print(f'  violation rate mean = {st.mean(rates):.4f}  sd = {st.stdev(rates) if len(rates) > 1 else 0.0:.4f}')

    summary = {
        'per_seed': {str(k): v for k, v in ditto_results_by_seed.items()},
        'aggregate_violation_rate': st.mean(rates),
        'aggregate_sd': st.stdev(rates) if len(rates) > 1 else 0.0,
        'seeds': DITTO_SEEDS,
        'source': DITTO_SOURCE, 'target': DITTO_TARGET, 'k': DITTO_K,
    }
    summary_path = Path(DECISIONS_ROOT) / 'ditto_pairwise_k100' / 'summary.json'
    summary_path.write_text(json.dumps(summary, indent=2))
    print(f'[saved] -> {summary_path}')

print('\n=== Three-paradigm comparison at rung 0 (WDC-Watches) ===')
print(f'  LLM K=2 random on LLaMA-3.3-70B:            0.1436 ± 0.0189  (from cell 8)')
if rates:
    print(f'  Ditto WS K=100 pairwise-only:               {st.mean(rates):.4f} ± {st.stdev(rates) if len(rates) > 1 else 0.0:.4f}  (measured here)')
print(f'  Ditto WS K=100 + union-find clustering:     0.0000 (by construction)')
